# 🎬 End-to-End Content-Based Movie Recommendation System

## 1. Project Overview & Architecture
Recommendation systems are foundational algorithms in modern digital platforms (e.g., Netflix, Spotify, Amazon, YouTube). They help users navigate massive item catalogs by estimating their preferences.

There are two primary paradigms in recommender systems:
1. **Collaborative Filtering:** Relies on past user behavior and interactions (e.g., user ratings, clicks, watch history) to find similarities between users or items.
2. **Content-Based Filtering:** Focuses on the inherent metadata/attributes of items (e.g., genre, cast, director, plot summary). If a user likes item $A$, the system recommends item $B$ which shares similar features.

In this project, we implement a **Content-Based Movie Recommender System** using the **TMDB 5,000 Movies & Credits** dataset.

### Pipeline Workflow:
1. **Data Ingestion & Inspection:** Ingest movies and credits datasets.
2. **Merging & Feature Selection:** Combine tables on movie title and isolate influential metadata.
3. **Data Cleaning & Handling Missing Values:** Handle missing entries and duplicates.
4. **Feature Extraction:** Parse structured JSON columns (`genres`, `keywords`, `cast`, `crew`).
5. **Entity Normalization:** Concatenate tokens (e.g., `Sam Worthington` $\rightarrow$ `SamWorthington`) to avoid token ambiguity in vector space.
6. **Natural Language Processing (NLP):** Apply tokenization, case-folding, and **Porter Stemming**.
7. **Vectorization:** Convert text tags into high-dimensional vectors via **Bag-of-Words (CountVectorizer)**.
8. **Similarity Computation:** Calculate pairwise **Cosine Similarity** between all movie vectors.
9. **Recommendation Engine:** Sort nearest items and retrieve the top $N$ recommendations.
10. **Model Serialization:** Export model artifacts (`.pkl`) for production deployment via **Streamlit**.


## 2. Environment Setup & Library Imports
We import standard scientific computing, data manipulation, and NLP libraries:
- `pandas` & `numpy` for data manipulation and array processing.
- `ast` (Abstract Syntax Tree) for safely evaluating Python literal expressions from stringified JSON columns.
- `nltk` (PorterStemmer) for word stemming.
- `scikit-learn` for CountVectorizer and Cosine Similarity calculation.
- `pickle` & `gzip` for serializing and compressing trained model artifacts for deployment.


In [1]:
import os
import ast
import gzip
import pickle
import numpy as np
import pandas as pd
import nltk
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")


Libraries imported successfully!


## 3. Data Ingestion & Initial Inspection
We load two datasets provided by The Movie Database (TMDb):
- `tmdb_5000_movies.csv`: Contains metadata such as budget, genres, homepage, id, keywords, overview, popularity, release date, revenue, runtime, vote average, and vote count.
- `tmdb_5000_credits.csv`: Contains movie_id, title, cast (actors, characters), and crew (directors, producers, writers, etc.).


In [2]:
# Load the CSV datasets
movies_df = pd.read_csv("tmdb_5000_movies.csv")
credits_df = pd.read_csv("tmdb_5000_credits.csv")

print(f"Movies dataset shape:  {movies_df.shape}")
print(f"Credits dataset shape: {credits_df.shape}")


Movies dataset shape:  (4803, 20)
Credits dataset shape: (4803, 4)


In [3]:
# Preview first two rows of movies dataframe
movies_df.head(2)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [4]:
# Preview first two rows of credits dataframe
credits_df.head(2)


,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


## 4. Merging Datasets & Feature Selection
Both tables share the unique movie identifier `title` (as well as `id`/`movie_id`). We merge them on the `title` column to combine cast/crew information with movie metadata into a single unified DataFrame.


In [5]:
# Merge both dataframes on 'title'
movies = movies_df.merge(credits_df, on="title")
print(f"Merged dataset shape: {movies.shape}")
movies.head(2)


Merged dataset shape: (4809, 23)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [6]:
# Inspect merged schema and data types
movies.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

### Feature Selection Rationale
In building a content-based recommendation system driven by qualitative text descriptors, not all columns are equally informative:
- **Columns kept:**
  - `movie_id` (or `id`): Needed for TMDB API poster lookup in production.
  - `title`: Primary movie title identifier.
  - `overview`: Narrative plot description containing vital thematic context.
  - `genres`: High-level genre categorization (Action, Sci-Fi, Comedy, etc.).
  - `keywords`: Deep, granular plot elements (space, alien, futuristic, superhero).
  - `cast`: Top lead actors who define movie star power.
  - `crew`: Director, whose cinematic style heavily dictates similarity.
- **Columns dropped:**
  - `budget`, `revenue`, `popularity`: Numerical variables that do not describe thematic content.
  - `status`, `runtime`, `vote_average`: Informative for ranking, but less suited for text tag similarity.


In [7]:
# Select the relevant feature columns
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
print(f"Selected features shape: {movies.shape}")
movies.head(2)


Selected features shape: (4809, 7)


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


## 5. Data Cleaning & Handling Missing / Duplicate Values
Before performing NLP operations, we verify data completeness and drop missing or duplicate entries.


In [8]:
# Check for missing values
print("Missing values per column:")
print(movies.isnull().sum())


Missing values per column:
movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64


In [9]:
# Drop rows with null values and check for duplicates
movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)
print(f"Number of duplicate rows: {movies.duplicated().sum()}")
print(f"Cleaned dataset shape: {movies.shape}")


Number of duplicate rows: 0
Cleaned dataset shape: (4806, 7)


## 6. Feature Extraction & Parsing JSON Attributes
Columns like `genres`, `keywords`, `cast`, and `crew` are stored as stringified JSON arrays of dictionaries (e.g., `'[{"id": 28, "name": "Action"}, ...]'`).

We parse these using Python's `ast.literal_eval`:
1. **Genres & Keywords:** Extract list of category/keyword names.
2. **Cast:** Extract names of the top 3 lead actors (most influential for recommendation).
3. **Crew:** Extract the movie **Director** (`job == 'Director'`).
4. **Overview:** Split the plot overview string into a list of word tokens.


In [10]:
# Helper function to extract 'name' values from stringified list of dictionaries
def convert(obj):
    names = []
    for item in ast.literal_eval(obj):
        names.append(item['name'])
    return names

# Apply conversion to genres and keywords
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies[['title', 'genres', 'keywords']].head(2)


,title,genres,keywords
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon..."
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ..."


In [11]:
# Helper function to extract top 3 lead actors
def convert_cast(obj, top_n=3):
    cast_names = []
    counter = 0
    for item in ast.literal_eval(obj):
        if counter < top_n:
            cast_names.append(item['name'])
            counter += 1
        else:
            break
    return cast_names

# Apply conversion to cast column
movies['cast'] = movies['cast'].apply(convert_cast)
movies[['title', 'cast']].head(2)


,title,cast
0,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]"
1,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]"


In [12]:
# Helper function to extract the Director from crew metadata
def fetch_director(obj):
    directors = []
    for item in ast.literal_eval(obj):
        if item.get('job') == 'Director':
            directors.append(item['name'])
            break
    return directors

# Apply conversion to crew column
movies['crew'] = movies['crew'].apply(fetch_director)
movies[['title', 'crew']].head(2)


,title,crew
0,Avatar,[James Cameron]
1,Pirates of the Caribbean: At World's End,[Gore Verbinski]


In [13]:
# Tokenize the overview string into a list of words
movies['overview'] = movies['overview'].apply(lambda x: x.split() if isinstance(x, str) else [])
movies[['title', 'overview']].head(2)


,title,overview
0,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."


## 7. Entity Normalization & Tag Construction

### The Entity Space Problem:
Consider two distinct individuals:
- `Sam Worthington` (actor in Avatar)
- `Sam Mendes` (director of Skyfall)

If spaces are left intact, the vectorizer tokenizes both into `'sam'`, `'worthington'`, and `'mendes'`. A search for movies by `Sam Mendes` would inadvertently trigger similarity with `Sam Worthington` because of the shared token `'sam'`.

**Solution:** Remove whitespace within entities so each name becomes a unique single token:
- `'Sam Worthington'` $\rightarrow$ `'SamWorthington'`
- `'Sam Mendes'` $\rightarrow$ `'SamMendes'`
- `'Science Fiction'` $\rightarrow$ `'ScienceFiction'`


In [14]:
# Remove spaces within names/phrases across categorical columns
def collapse_spaces(items):
    return [item.replace(" ", "") for item in items]

movies['genres'] = movies['genres'].apply(collapse_spaces)
movies['keywords'] = movies['keywords'].apply(collapse_spaces)
movies['cast'] = movies['cast'].apply(collapse_spaces)
movies['crew'] = movies['crew'].apply(collapse_spaces)

movies[['genres', 'keywords', 'cast', 'crew']].head(2)


,genres,keywords,cast,crew
0,"[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,"[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]


### Combining All Features into a Unified `tags` Column
Now we concatenate `overview`, `genres`, `keywords`, `cast`, and `crew` into a single comprehensive token list, and then join them into a lowercase text string.


In [15]:
# Concatenate all token lists into unified 'tags'
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# Create lean dataframe for vectorization
new_data = movies[['movie_id', 'title', 'tags']].copy()
new_data['id'] = new_data['movie_id']  # support both 'id' and 'movie_id'

# Convert token lists into space-separated lowercase strings
new_data['tags'] = new_data['tags'].apply(lambda x: " ".join(x).lower())

print(f"Processed dataset shape: {new_data.shape}")
new_data.head(2)


Processed dataset shape: (4806, 4)


,movie_id,title,tags,id
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di...",19995
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha...",285


## 8. Natural Language Processing: Porter Stemming
In English, words occur in various inflectional and derived forms (e.g., *action*, *actions*, *acting*).

If left untreated, `CountVectorizer` treats each form as a completely separate feature dimension.

We apply the **Porter Stemmer** from `nltk` to reduce all words to their linguistic root stem:
- `actions` $\rightarrow$ `action`
- `loving`, `loved` $\rightarrow$ `love`


In [16]:
# Initialize PorterStemmer and demonstrate on sample words
ps = PorterStemmer()

sample_words = ["actions", "action", "activities", "directed", "directing", "dancer", "dancing"]
for word in sample_words:
    print(f"'{word}' -> stem: '{ps.stem(word)}'")


'actions' -> stem: 'action'
'action' -> stem: 'action'
'activities' -> stem: 'activ'
'directed' -> stem: 'direct'
'directing' -> stem: 'direct'
'dancer' -> stem: 'dancer'
'dancing' -> stem: 'danc'


In [17]:
# Define stemming function and apply across all tags
def stem(text):
    stemmed_words = [ps.stem(word) for word in text.split()]
    return " ".join(stemmed_words)

new_data['tags'] = new_data['tags'].apply(stem)
print("Stemmed tags for first movie (Avatar):")
print(new_data['tags'].iloc[0][:350] + "...")


Stemmed tags for first movie (Avatar):
in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powe...


## 9. Text Vectorization using CountVectorizer (Bag-of-Words)
To compute mathematical similarity, text must be converted into numerical vectors in Euclidean space.

We use **CountVectorizer**:
- `max_features=5000`: Restricts vocabulary to the 5,000 most frequent informative words, removing noise while keeping feature dimensionality efficient.
- `stop_words='english'`: Automatically filters out common grammatical words (*the*, *is*, *at*, *which*, *on*).


In [18]:
# Initialize CountVectorizer and transform tags
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_data['tags']).toarray()

print(f"Feature vectors matrix shape: {vectors.shape}")
print(f"Matrix memory size: {vectors.nbytes / (1024 * 1024):.2f} MB")


Feature vectors matrix shape: (4806, 5000)
Matrix memory size: 183.33 MB


In [19]:
# Inspect sample vocabulary features learned by vectorizer
feature_names = cv.get_feature_names_out()
print(f"Total vocabulary terms: {len(feature_names)}")
print("Sample 30 features:", feature_names[200:230])


Total vocabulary terms: 5000
Sample 30 features: ['amus' 'amusementpark' 'amy' 'amyadam' 'amysmart' 'ana' 'anakin'
 'analyst' 'anarchiccomedi' 'ancient' 'ancientrom' 'ancientworld'
 'anderson' 'andi' 'andiemacdowel' 'andrew' 'android' 'andy' 'andygarcía'
 'angel' 'angela' 'angelabassett' 'angeles' 'angelinajoli' 'anger' 'angle'
 'angri' 'ani' 'anim' 'animalattack']


## 10. Calculating Pairwise Cosine Similarity

### Mathematical Foundation:
In high-dimensional text vector spaces, Euclidean distance (magnitude-based) can be misleading because longer descriptions naturally have larger magnitudes.

Instead, we compute **Cosine Similarity**, which measures the cosine of the angle between two non-zero vectors:

$$\text{Cosine Similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \sqrt{\sum_{i=1}^{n} B_i^2}}$$

- **Score Range:** $[0, 1]$ (where 1 means identical orientation/maximum similarity, and 0 means orthogonal/no common terms).
- For $4,806$ movies, the resulting similarity matrix is of shape $(4806, 4806)$.


In [20]:
# Compute pairwise cosine similarity matrix
similarity_measure = cosine_similarity(vectors).astype(np.float32)

print(f"Similarity matrix shape: {similarity_measure.shape}")
print(f"Data type: {similarity_measure.dtype}")
print(f"Memory size (float32): {similarity_measure.nbytes / (1024 * 1024):.2f} MB")


Similarity matrix shape: (4806, 4806)
Data type: float32
Memory size (float32): 88.11 MB


In [21]:
# Inspect self-similarity and sample pairwise scores for the first movie (Avatar)
print(f"Self-similarity score for Avatar: {similarity_measure[0][0]:.4f}")
print("First 10 similarity scores for Avatar with other movies:")
print(np.round(similarity_measure[0][:10], 4))


Self-similarity score for Avatar: 1.0000
First 10 similarity scores for Avatar with other movies:
[1.     0.0835 0.086  0.0735 0.1893 0.1084 0.0402 0.1467 0.0592 0.0978]


## 11. Developing & Testing the Recommendation Function
Given a query movie title:
1. Locate its index in `new_data`.
2. Extract its similarity row vector from `similarity_measure`.
3. Enumerate each movie index with its similarity score: `(index, score)`.
4. Sort descending based on score.
5. Retrieve top $N$ movies (ignoring index 0, which is the movie itself).


In [22]:
def recommend(movie, top_n=5):
    """
    Recommend top_n most similar movies for a given movie title.
    """
    if movie not in new_data['title'].values:
        print(f"❌ Movie '{movie}' not found in catalog.")
        return []

    movie_index = new_data[new_data['title'] == movie].index[0]
    distances = similarity_measure[movie_index]
    sorted_movies = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:top_n + 1]

    print(f"\n🎯 Top {top_n} Recommendations for '{movie}':")
    print("-" * 50)
    recommendations = []
    for rank, (idx, score) in enumerate(sorted_movies, 1):
        rec_title = new_data.iloc[idx].title
        recommendations.append(rec_title)
        print(f"{rank}. {rec_title:<35} (Score: {score:.4f})")
    return recommendations


In [23]:
# Test Case 1: Avatar (Sci-Fi / Action / Alien World)
recommend('Avatar')



🎯 Top 5 Recommendations for 'Avatar':
--------------------------------------------------
1. Aliens vs Predator: Requiem         (Score: 0.2868)
2. Aliens                              (Score: 0.2690)
3. Falcon Rising                       (Score: 0.2605)
4. Independence Day                    (Score: 0.2556)
5. Titan A.E.                          (Score: 0.2504)


['Aliens vs Predator: Requiem',
 'Aliens',
 'Falcon Rising',
 'Independence Day',
 'Titan A.E.']

In [24]:
# Test Case 2: The Dark Knight (Crime / Action / Superhero)
recommend('The Dark Knight')



🎯 Top 5 Recommendations for 'The Dark Knight':
--------------------------------------------------
1. The Dark Knight Rises               (Score: 0.4226)
2. Batman Begins                       (Score: 0.4022)
3. Batman Returns                      (Score: 0.3314)
4. Batman Forever                      (Score: 0.2941)
5. Batman                              (Score: 0.2767)


['The Dark Knight Rises',
 'Batman Begins',
 'Batman Returns',
 'Batman Forever',
 'Batman']

In [25]:
# Test Case 3: The Avengers (Marvel / Superhero Action)
recommend('The Avengers')



🎯 Top 5 Recommendations for 'The Avengers':
--------------------------------------------------
1. Iron Man 3                          (Score: 0.3181)
2. Avengers: Age of Ultron             (Score: 0.3179)
3. Captain America: Civil War          (Score: 0.3078)
4. Captain America: The First Avenger  (Score: 0.2870)
5. Iron Man                            (Score: 0.2711)


['Iron Man 3',
 'Avengers: Age of Ultron',
 'Captain America: Civil War',
 'Captain America: The First Avenger',
 'Iron Man']

## 12. Model Serialization for Production Deployment
To deploy this recommendation engine in a live web application (Streamlit), we serialize the precomputed data structures:
- `movie_dict.pkl` / `movies.pkl`: Clean movie metadata with IDs and titles.
- `similarity.pkl`: Precomputed similarity matrix.
- `similarity.pkl.gz`: Gzip-compressed similarity matrix (~36 MB) for ultra-fast loading and Git repository compatibility (< 100 MB).


In [26]:
# Export movie dictionary
with open('movie_dict.pkl', 'wb') as f:
    pickle.dump(new_data.to_dict(), f, protocol=pickle.HIGHEST_PROTOCOL)

# Export movie DataFrame
with open('movies.pkl', 'wb') as f:
    pickle.dump(new_data, f, protocol=pickle.HIGHEST_PROTOCOL)

# Export compressed similarity matrix
with gzip.open('similarity.pkl.gz', 'wb') as f:
    pickle.dump(similarity_measure, f, protocol=pickle.HIGHEST_PROTOCOL)

# Export uncompressed similarity matrix
with open('similarity.pkl', 'wb') as f:
    pickle.dump(similarity_measure, f, protocol=pickle.HIGHEST_PROTOCOL)

print("✅ Model artifacts successfully exported:")
print(f"  - movie_dict.pkl    ({os.path.getsize('movie_dict.pkl') / (1024*1024):.2f} MB)")
print(f"  - movies.pkl        ({os.path.getsize('movies.pkl') / (1024*1024):.2f} MB)")
print(f"  - similarity.pkl.gz ({os.path.getsize('similarity.pkl.gz') / (1024*1024):.2f} MB)")
print(f"  - similarity.pkl    ({os.path.getsize('similarity.pkl') / (1024*1024):.2f} MB)")


✅ Model artifacts successfully exported:
  - movie_dict.pkl    (2.14 MB)
  - movies.pkl        (2.13 MB)
  - similarity.pkl.gz (36.49 MB)
  - similarity.pkl    (88.11 MB)


## 13. Summary & Next Steps
We have built a content-based recommendation model from scratch:
- **Cleaned and unified** 4,806 movies from TMDb.
- **Engineered tags** capturing genres, keywords, overview, cast, and director.
- **Normalized and stemmed** text to eradicate token ambiguity.
- **Computed cosine similarity** over 5,000-dimensional vector space.
- **Serialized artifacts** for deployment in `app.py`.

### Web Application:
Run the Streamlit application via:
```bash
streamlit run app.py
```
